In [ ]:
"""
Python-based Google Cluster Data Parser
Replicates AGOCS functionality for constraint extraction
"""

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import gzip
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class GCDParser:
    """
    Parse Google Cluster Data traces and extract task constraints
    Similar to AGOCS framework functionality
    """
    
    def __init__(self, gcd_path: str):
        """
        Initialize parser with path to GCD data
        
        Args:
            gcd_path: Path to directory containing GCD CSV files
        """
        self.gcd_path = Path(gcd_path)
        self.task_constraints = None
        self.machine_attributes = None
        self.task_events = None
        
    def load_task_constraints(self, 
                             chunk_size: int = 100000,
                             max_rows: Optional[int] = None) -> pd.DataFrame:
        """
        Load and parse task_constraints table
        
        GCD Schema:
        - time: timestamp
        - job ID: job identifier
        - task index: task identifier within job
        - comparison operator: type of constraint
        - value: constraint value
        - constraint name: type of constraint (3 = node affinity)
        """
        print("Loading task constraints...")
        
        constraint_file = self.gcd_path / "task_constraints.csv.gz"
        
        if not constraint_file.exists():
            constraint_file = self.gcd_path / "task_constraints.csv"
        
        if not constraint_file.exists():
            raise FileNotFoundError(f"Cannot find task_constraints file in {self.gcd_path}")
        
        # Column names for GCD task_constraints
        columns = [
            'timestamp',
            'job_id',
            'task_index',
            'comparison_operator',
            'value',
            'constraint_name'
        ]
        
        chunks = []
        total_rows = 0
        
        try:
            # Try reading compressed file
            for chunk in pd.read_csv(
                constraint_file,
                names=columns,
                compression='gzip' if str(constraint_file).endswith('.gz') else None,
                chunksize=chunk_size
            ):
                chunks.append(chunk)
                total_rows += len(chunk)
                
                if max_rows and total_rows >= max_rows:
                    break
                    
                print(f"Loaded {total_rows:,} constraints...", end='\r')
        
        except Exception as e:
            print(f"\nError loading file: {e}")
            raise
        
        self.task_constraints = pd.concat(chunks, ignore_index=True)
        print(f"\nTotal constraints loaded: {len(self.task_constraints):,}")
        
        return self.task_constraints
    
    def load_machine_attributes(self, 
                               chunk_size: int = 100000) -> pd.DataFrame:
        """
        Load machine attributes table
        
        GCD Schema:
        - time: timestamp
        - machine ID: machine identifier
        - attribute name: attribute type
        - attribute value: attribute value
        - deleted: whether attribute was deleted
        """
        print("Loading machine attributes...")
        
        attr_file = self.gcd_path / "machine_attributes.csv.gz"
        
        if not attr_file.exists():
            attr_file = self.gcd_path / "machine_attributes.csv"
        
        if not attr_file.exists():
            print("Warning: machine_attributes file not found")
            return None
        
        columns = [
            'timestamp',
            'machine_id',
            'attribute_name',
            'attribute_value',
            'deleted'
        ]
        
        chunks = []
        
        for chunk in tqdm(
            pd.read_csv(
                attr_file,
                names=columns,
                compression='gzip' if str(attr_file).endswith('.gz') else None,
                chunksize=chunk_size
            ),
            desc="Loading machine attributes"
        ):
            chunks.append(chunk)
        
        self.machine_attributes = pd.concat(chunks, ignore_index=True)
        print(f"Total machine attributes loaded: {len(self.machine_attributes):,}")
        
        return self.machine_attributes
    
    def filter_node_affinity_tasks(self) -> pd.DataFrame:
        """
        Filter for tasks with node affinity constraints (constraint_name == 3)
        This matches the paper's focus on tasks with specific node requirements
        """
        if self.task_constraints is None:
            raise ValueError("Must load task_constraints first")
        
        print("Filtering for node affinity constraints...")
        
        # Constraint name 3 represents node affinity in GCD
        node_affinity = self.task_constraints[
            self.task_constraints['constraint_name'] == 3
        ].copy()
        
        print(f"Tasks with node affinity: {len(node_affinity):,}")
        
        return node_affinity
    
    def identify_constrained_tasks(self, 
                                  max_nodes: int = 1000) -> Dict[str, pd.DataFrame]:
        """
        Identify tasks that can run on:
        1. Exactly 1 node
        2. Fewer than max_nodes (default 1000)
        
        This matches the paper's classification
        """
        if self.task_constraints is None:
            raise ValueError("Must load task_constraints first")
        
        print("Identifying constrained tasks...")
        
        # Get node affinity constraints
        node_affinity = self.filter_node_affinity_tasks()
        
        # Count unique machines per task
        task_node_counts = node_affinity.groupby(
            ['job_id', 'task_index']
        )['value'].agg([
            ('num_nodes', 'nunique'),
            ('node_list', lambda x: list(x.unique()))
        ]).reset_index()
        
        # Single node tasks
        single_node_tasks = task_node_counts[
            task_node_counts['num_nodes'] == 1
        ].copy()
        
        # Limited node tasks (< max_nodes)
        limited_node_tasks = task_node_counts[
            (task_node_counts['num_nodes'] > 1) &
            (task_node_counts['num_nodes'] < max_nodes)
        ].copy()
        
        print(f"\nTask Classification:")
        print(f"  Single-node tasks: {len(single_node_tasks):,}")
        print(f"  Limited-node tasks (<{max_nodes}): {len(limited_node_tasks):,}")
        print(f"  Total constrained tasks: {len(task_node_counts):,}")
        
        return {
            'single_node': single_node_tasks,
            'limited_node': limited_node_tasks,
            'all_constrained': task_node_counts
        }
    
    def create_task_features(self, 
                           tasks_df: pd.DataFrame) -> pd.DataFrame:
        """
        Create compacted constraint features for tasks
        Similar to AGOCS preprocessing
        """
        print("Creating task features...")
        
        if self.task_constraints is None:
            raise ValueError("Must load task_constraints first")
        
        # Merge task constraints with task list
        features = tasks_df.merge(
            self.task_constraints,
            on=['job_id', 'task_index'],
            how='left'
        )
        
        # Create compact constraint representation
        # Format: constraint_name_operator_value
        features['constraint_string'] = features.apply(
            lambda row: f"{row['constraint_name']}_{row['comparison_operator']}_{row['value']}",
            axis=1
        )
        
        # Group constraints by task
        task_features = features.groupby(['job_id', 'task_index']).agg({
            'constraint_string': lambda x: '|'.join(sorted(set(x))),
            'constraint_name': lambda x: list(x),
            'comparison_operator': lambda x: list(x),
            'value': lambda x: list(x)
        }).reset_index()
        
        return task_features
    
    def export_for_ml(self, 
                     output_path: str,
                     max_nodes: int = 1000):
        """
        Export processed data ready for ML training
        Matches the paper's preprocessing pipeline
        """
        print("\n=== Exporting ML-ready dataset ===")
        
        # Get constrained tasks
        constrained_tasks = self.identify_constrained_tasks(max_nodes)
        
        # Create features for single-node tasks
        single_node_features = self.create_task_features(
            constrained_tasks['single_node']
        )
        
        # Create features for limited-node tasks
        limited_node_features = self.create_task_features(
            constrained_tasks['limited_node']
        )
        
        # Add labels
        single_node_features['task_type'] = 'single_node'
        limited_node_features['task_type'] = 'limited_node'
        
        # Combine
        ml_dataset = pd.concat([
            single_node_features,
            limited_node_features
        ], ignore_index=True)
        
        # Save
        output_path = Path(output_path)
        ml_dataset.to_csv(output_path / 'ml_ready_dataset.csv', index=False)
        
        # Save metadata
        metadata = {
            'total_tasks': len(ml_dataset),
            'single_node_tasks': len(single_node_features),
            'limited_node_tasks': len(limited_node_features),
            'max_nodes_threshold': max_nodes,
            'unique_constraints': ml_dataset['constraint_string'].nunique()
        }
        
        pd.Series(metadata).to_csv(output_path / 'dataset_metadata.csv')
        
        print(f"\nDataset exported to: {output_path}")
        print(f"Total samples: {len(ml_dataset):,}")
        
        return ml_dataset


def main():
    """
    Example usage
    """
    # Path to your GCD data
    gcd_path = "./google_cluster_data_2011"
    
    # Initialize parser
    parser = GCDParser(gcd_path)
    
    # Load data
    parser.load_task_constraints(max_rows=1000000)  # Limit for testing
    
    # Optional: load machine attributes
    # parser.load_machine_attributes()
    
    # Export ML-ready dataset
    ml_data = parser.export_for_ml(
        output_path="./processed_data",
        max_nodes=1000
    )
    
    print("\n✓ Processing complete!")
    return ml_data


if __name__ == "__main__":
    ml_data = main()


# 2. Download Google Cluster Data
# Visit: https://github.com/google/cluster-data
# Download the 2011 traces (ClusterData2011_2)

# 3. Extract the data
mkdir google_cluster_data_2011
cd google_cluster_data_2011
# Extract your downloaded files here

# 4. Run the parser
python gcd_parser.py